# LELA60341: Research Methods in Computational and Corpus Linguistics Week 8

## A logistic regression model, built from parts (after class)

In class you built a `Linear` layer, the loss functions `bce_loss` and `bce_grad`, and a training loop, and trained a logistic regression model on the simulated data from LELA60331 Week 7. In this workbook you use the model, evaluate it, and compare several models side by side.

| Part | Topic |
|---|---|
| 0 | Set-up: everything from class |
| 1 | Using and evaluating the model |
| 2 | Several models side by side |

Read each explanation, run the example cells, then try the 🏋️ exercises. In an exercise, replace each `???` with your own code, then run the ✅ check cell underneath. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell. Parts marked ⭐ **Challenge** are optional.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 0 · Set-up

The next cells set up everything from the in-class workbook, so that this workbook works on its own: the simulated data `X` and `y`, the helper functions, the `Linear` class, `bce_loss` and `bce_grad`. Read through them to remind yourself of each piece, then run them.

In [ ]:
import math
import random
import matplotlib.pyplot as plt

random.seed(10)
x = [[], []]
y = []
for i in range(50):
    if random.random() > 0.5:          # a negative text: fewer positive than negative words
        x[0].append(random.gauss(2, 1))
        x[1].append(random.gauss(3, 1))
        y.append(0)
    else:                              # a positive text: more positive than negative words
        x[0].append(random.gauss(3, 1))
        x[1].append(random.gauss(2, 1))
        y.append(1)
X = [[x[0][k], x[1][k]] for k in range(len(y))]

In [ ]:
import math
import random
import matplotlib.pyplot as plt

def dot(a, b):
    # the dot product of two lists of the same length
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def mat_vec(matrix, vector):
    # the dot product of each row of the matrix with the vector
    return [dot(row, vector) for row in matrix]

def transpose(matrix):
    # the rows of the matrix turned into columns
    return [[row[j] for row in matrix] for j in range(len(matrix[0]))]

def sigmoid(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    else:
        return math.exp(z) / (1 + math.exp(z))      # the same value, without calculating math.exp(-z)

class Linear:
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        self.weight = [[random.random() for j in range(n_in)] for c in range(n_out)]
        self.bias = [random.random() for c in range(n_out)]

    def forward(self, X):
        self.inputs = X                       # kept for the backward pass
        scores = []
        for x in X:
            z = mat_vec(self.weight, x)
            for c in range(len(z)):
                z[c] += self.bias[c]
            scores.append(z)
        return scores

    def backward(self, grad):
        n_out = len(self.weight)
        n_in = len(self.weight[0])
        self.weight_grad = [[0] * n_in for c in range(n_out)]
        self.bias_grad = [0] * n_out
        for k in range(len(self.inputs)):
            for c in range(n_out):
                for j in range(n_in):
                    self.weight_grad[c][j] += grad[k][c] * self.inputs[k][j]
                self.bias_grad[c] += grad[k][c]
        # the gradients for the inputs, for a layer that comes before this one
        return [mat_vec(transpose(self.weight), g) for g in grad]

    def update(self, lr):
        for c in range(len(self.weight)):
            for j in range(len(self.weight[c])):
                self.weight[c][j] -= lr * self.weight_grad[c][j]
            self.bias[c] -= lr * self.bias_grad[c]

def bce_loss(scores, y):
    # the average cross-entropy loss for two classes, from the scores
    eps = 0.00001
    losses = []
    for k in range(len(y)):
        q = sigmoid(scores[k][0])
        if y[k] == 1:
            losses.append(-math.log2(q + eps))
        else:
            losses.append(-math.log2(1 - q + eps))
    return sum(losses) / len(losses)

def bce_grad(scores, y):
    # the gradient of the loss for each score: the error divided by the number of examples
    n = len(y)
    return [[(sigmoid(scores[k][0]) - y[k]) / n] for k in range(n)]

Finally, the training from class: a `Linear` layer with 2 inputs and 1 output, trained for 1000 epochs with a learning rate of 0.5. Run the cell: the final loss should be 0.656, as in class.

In [ ]:
random.seed(10)
model = Linear(2, 1)
loss_history = []
for epoch in range(1000):
    scores = model.forward(X)
    loss = bce_loss(scores, y)
    loss_history.append(loss)
    model.backward(bce_grad(scores, y))
    model.update(0.5)
print(model.weight, model.bias)
print("final loss:", round(loss_history[-1], 3))

---
# Part 1 · Using and evaluating the model

The model gives **scores**. To use it, we write three small functions. They work for any model that has a `forward` method and gives one score per example, so they are kept outside the class.

### 🏋️ Exercise 1 · Practice — Probabilities and predictions

1. Complete `predict_proba(model, X)`: the probability that each example is positive, the sigmoid of its score. *Hint:* a list comprehension over the rows of `model.forward(X)`; each row holds one score.
2. Complete `predict(model, X)`: the predicted class of each example, 1 if its probability is above 0.5 and 0 otherwise
3. `new_probs`: the probability that each of these two new texts is positive (LELA60331 Week 7, Activity 3). Remember that the features are **log** counts, and that the model takes a **list** of examples.
   * a) 2 positive words and 3 negative words
   * b) 10 positive words and 1 negative word

In [ ]:
def predict_proba(model, X):
    return ???          # 1.

def predict(model, X):
    return ???        # 2.

new_probs = ???    # 3.
print(new_probs)                                 # about 0.466 and 0.961
print(predict(model, X[:10]), y[:10])

In [ ]:
# ✅ Check
check("1. predict_proba", lambda: abs(predict_proba(model, [[0, 0]])[0] - sigmoid(model.bias[0])) < 1e-12)
check("2. predict", lambda: predict(model, X[:5]) == [int(p > 0.5) for p in predict_proba(model, X[:5])])
check("3. new_probs", lambda: abs(new_probs[0] - 0.466) < 0.005 and abs(new_probs[1] - 0.961) < 0.005)

### 🏋️ Exercise 2 · Practice — `evaluate`

Complete `evaluate(model, X, y)`, which returns a **dict** (Week 2) with the accuracy, precision and recall (LELA60331 Week 7, Activities 2 and 4):

1. the predicted classes, with `predict`
2. the conditions for false positives and false negatives (the others are done for you)
3. the three entries of the dict

In [ ]:
def evaluate(model, X, y):
    predicted = ???                         # 1.
    correct = 0
    tp = 0
    fp = 0
    fn = 0
    for k in range(len(y)):
        if predicted[k] == y[k]:
            correct += 1
        if predicted[k] == 1 and y[k] == 1:
            tp += 1
        if ???:               # 2. predicted positive, but really negative
            fp += 1
        if ???:               #    predicted negative, but really positive
            fn += 1
    return {"accuracy": ???, "precision": ???, "recall": ???}     # 3.

results = evaluate(model, X, y)
print(results)                     # accuracy 0.78, precision 0.75, recall 0.783

In [ ]:
# ✅ Check
check("accuracy 0.78, precision 0.75, recall 0.783",
      lambda: results["accuracy"] == 0.78 and results["precision"] == 0.75 and abs(results["recall"] - 0.783) < 0.001)

---
# Part 2 · Several models side by side

In a notebook written as a series of cells, trying a different learning rate means changing `lr` and re-running the training cell, which replaces the old model. With a class, each model is a separate object, so you can keep several and compare them.

### 🏋️ Exercise 3 · Practice — A `train` function

Complete `train(model, X, y, lr, n_epochs)`, which trains `model` with the loop from Exercise 11 of the in-class workbook and returns its loss history. The test underneath checks that it gives the same result as Exercise 11 of the in-class workbook.

In [ ]:
def train(model, X, y, lr, n_epochs):
    history = []
    for epoch in range(n_epochs):
        scores = ???
        history.append(???)
        ???
        ???
    return history

random.seed(10)
test_model = Linear(2, 1)
test_history = train(test_model, X, y, 0.5, 1000)
print(round(test_history[-1], 3))         # the same as the training loop: 0.656

In [ ]:
# ✅ Check
check("train gives the same as the training loop", lambda: abs(test_history[-1] - loss_history[-1]) < 1e-12 and len(test_history) == 1000)

### 🏋️ Exercise 4 · Practice — Comparing learning rates

1. `models`: a dict from each learning rate in `rates` to a new `Linear(2, 1)` model (made after `random.seed(10)`, so that they all start the same)
2. `histories`: a dict from each learning rate to the loss history from training its model with `train`

The cell then plots the three loss histories on the same axes. 🔮 **Predict** before you run it: what will each curve look like? (Think about what you saw when you changed the learning rate for linear regression in LELA60331 Week 4.)

In [ ]:
rates = [0.01, 0.5, 2]
models = {}
histories = {}
for lr in rates:
    random.seed(10)
    models[lr] = ???                                         # 1.
    histories[lr] = ???                 # 2.

for lr in rates:
    plt.plot(histories[lr], label="lr = " + str(lr))
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()
for lr in rates:
    print(lr, round(histories[lr][-1], 3), evaluate(models[lr], X, y))

In [ ]:
# ✅ Check
check("1. three models", lambda: sorted(models) == rates and all(isinstance(m, Linear) for m in models.values()))
check("2. each trained for 1000 epochs", lambda: all(len(h) == 1000 for h in histories.values()))
check("lr = 0.5 gives the same as the training loop", lambda: abs(histories[0.5][-1] - loss_history[-1]) < 1e-9)
check("the first model is unchanged", lambda: abs(bce_loss(model.forward(X), y) - 0.656) < 0.001)

⭐ **Challenge (optional):** add a method `__str__` to `Linear` (Week 7, Part 3), so that `print(model)` shows its weights and biases, rounded to 2 decimal places. *Hint:* copy the class from the set-up cell into a new cell, and add the method.

---
## ✏️ Summary

You can now:

* use a trained model to give probabilities and predictions for new examples
* evaluate a model with a function that works for any model with a `forward` method
* wrap the training loop in a function, and use it to keep several models side by side and compare them